# Retarded Fields: Validation

Validates `retarded_fields.py`'s `lw_fields` (Lienard-Wiechert E and B
fields for a point charge on an arbitrary path) three ways, in increasing
order of how much of the machinery each one exercises:

1. **Static charge**: must reduce to exact Coulomb.
2. **Uniformly-moving charge**: checked against the known closed form
   expressed via the *present* position -- a genuinely different-looking
   formula from the retarded-position LW fields, so matching it validates
   the retarded-time root-find together with the field formula.
3. **Accelerating charge**: total radiated power (Poynting flux
   integrated over a large sphere) checked against the Larmor formula.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import retarded_fields as rf

os.makedirs('media', exist_ok=True)
c = 1.0
eps0 = 1.0
q = 1.0

## Static charge -> exact Coulomb

In [2]:
r0 = np.array([0.0, 0.0, 0.0])
r_func = lambda t: r0
v_func = lambda t: np.zeros(3)
a_func = lambda t: np.zeros(3)

x = np.array([2.0, 1.0, 0.5])
t = 5.0
E, B = rf.lw_fields(r_func, v_func, a_func, x, t, q, c=c, eps0=eps0)

R = x - r0
E_coulomb = q / (4 * np.pi * eps0) * R / np.linalg.norm(R)**3
rel_err = np.linalg.norm(E - E_coulomb) / np.linalg.norm(E_coulomb)
print(f'E relative error vs. Coulomb: {rel_err:.2e}')
print(f'|B| (should be exactly 0 for a static charge): {np.linalg.norm(B):.2e}')
assert rel_err < 1e-10, 'static-charge field does not match Coulomb\'s law'
assert np.linalg.norm(B) < 1e-10, 'a static charge should have zero magnetic field'
print('PASS: static charge reduces exactly to Coulomb\'s law.')

E relative error vs. Coulomb: 1.31e-16
|B| (should be exactly 0 for a static charge): 0.00e+00
PASS: static charge reduces exactly to Coulomb's law.


## Uniformly-moving charge -> boosted Coulomb field

A charge moving at constant (mildly relativistic, `beta=0.3`) velocity
has a well-known closed-form field expressed in terms of its **present**
position (Griffiths eq. 10.75) -- notably *not* the retarded position the
Lienard-Wiechert formula is written in terms of. Matching this is a real
test that the retarded-time solve is correct, not just that the
zero-velocity limit works.

In [3]:
v0 = np.array([0.3, 0.0, 0.0])
r_func = lambda t: v0 * t
v_func = lambda t: v0
a_func = lambda t: np.zeros(3)

t = 3.0
x = np.array([2.5, 1.5, 0.7])
E, B = rf.lw_fields(r_func, v_func, a_func, x, t, q, c=c, eps0=eps0)

R_vec = x - r_func(t)  # PRESENT separation, not retarded
R_mag = np.linalg.norm(R_vec)
beta = np.linalg.norm(v0) / c
costheta = np.dot(R_vec, v0 / np.linalg.norm(v0)) / R_mag
sin2theta = 1 - costheta**2
E_present = q / (4 * np.pi * eps0) * (1 - beta**2) * R_vec / (R_mag**3 * (1 - beta**2 * sin2theta)**1.5)

rel_err = np.linalg.norm(E - E_present) / np.linalg.norm(E_present)
print(f'E relative error vs. present-position closed form: {rel_err:.2e}')
assert rel_err < 1e-10, 'uniformly-moving charge field does not match the known closed form'
print('PASS: uniformly-moving charge field matches the present-position closed form.')

E relative error vs. present-position closed form: 2.71e-16
PASS: uniformly-moving charge field matches the present-position closed form.


## Accelerating charge -> Larmor radiated power

A non-relativistic charge in circular motion (`v/c` tiny) has constant
acceleration *magnitude* at all times, so the Larmor-formula prediction
`P = q^2 a^2 / (6 pi eps0 c^3)` is time-independent -- convenient, since
it means observation points on the integration sphere don't need
synchronized retarded times. Points are placed on a Fibonacci sphere at a
radius many wavelengths out (checked explicitly) so the fields are
genuinely in the radiation zone.

In [4]:
r_orbit = 0.01
Omega = 0.05
v_speed = Omega * r_orbit
print(f'v/c = {v_speed:.2e} (non-relativistic)')

r_func = lambda t: np.array([r_orbit * np.cos(Omega * t), r_orbit * np.sin(Omega * t), 0.0])
v_func = lambda t: np.array([-r_orbit * Omega * np.sin(Omega * t), r_orbit * Omega * np.cos(Omega * t), 0.0])
a_func = lambda t: -Omega**2 * r_func(t)

a_mag = r_orbit * Omega**2
P_larmor = q**2 * a_mag**2 / (6 * np.pi * eps0 * c**3)

wavelength = 2 * np.pi * c / Omega
R_obs = 500.0
print(f'observation radius / wavelength = {R_obs / wavelength:.1f} (want several -- genuinely far field)')

mu0 = 1 / (eps0 * c**2)
N_pts = 800
i = np.arange(N_pts)
golden = (1 + 5**0.5) / 2
theta_pol = np.arccos(1 - 2 * (i + 0.5) / N_pts)
phi_azim = 2 * np.pi * i / golden
X = R_obs * np.sin(theta_pol) * np.cos(phi_azim)
Y = R_obs * np.sin(theta_pol) * np.sin(phi_azim)
Z = R_obs * np.cos(theta_pol)
domega = 4 * np.pi / N_pts  # equal-area weight for a Fibonacci sphere

P_total = 0.0
for k in range(N_pts):
    x = np.array([X[k], Y[k], Z[k]])
    E, B = rf.lw_fields(r_func, v_func, a_func, x, 0.0, q, c=c, eps0=eps0)
    S = np.cross(E, B) / mu0
    n_hat = x / np.linalg.norm(x)
    P_total += np.dot(S, n_hat) * R_obs**2 * domega

rel_err = abs(P_total - P_larmor) / P_larmor
print(f'P_larmor = {P_larmor:.4e}')
print(f'P_numeric (sphere integral) = {P_total:.4e}')
print(f'relative error: {rel_err:.2e}')
assert rel_err < 0.01, 'radiated power does not match the Larmor formula'
print('PASS: numerically integrated radiated power matches the Larmor formula.')

v/c = 5.00e-04 (non-relativistic)
observation radius / wavelength = 4.0 (want several -- genuinely far field)
P_larmor = 3.3157e-11
P_numeric (sphere integral) = 3.3157e-11
relative error: 6.58e-07
PASS: numerically integrated radiated power matches the Larmor formula.
